# Data Access from dCache

- Training: **High Performance Computing with Python and RS-DAT**, SURF (Utrecht)

- Date: March 19, 2026 

## 1. Introduction

[dCache](https://dcache.org/) is a massive storage system, an instance of which [is hosted at SURF](https://doc.grid.surfsara.nl/en/stable/Pages/Service/system_specifications/dcache_specs.html). dCache consists a heterogenous pool of server nodes (disk and tape) acting as a single virtual filesystem, which is designed to store and retrieve huge volumes of data.

dCache supports various APIs for data aceess, including HTTP/WebDAV, as well as username/password or token-based authentication.

In this notebook, we explore two datasets that we have downloaded/generated and that are currently hosted on dCache in the form of:

* A set of **NetCDF** files: (part of) the [Daymet](https://daac.ornl.gov/cgi-bin/dsviewer.pl?ds_id=1840) dataset, including daily weather data for North America (one NetCDF/HDF5 file per year and variable);
* A **Zarr** store: spring indices (date of first leaf and first bloom) derived from Daymet (more info [here](https://github.com/RS-DAT/Phenology)).

## 2. dCacheFS

In order to load data hosted on dCache directly into Python, we have built an adapter for the [Filesystem Spec (fsspec) library](https://filesystem-spec.readthedocs.io): [**dCacheFS**](https://dcachefs.readthedocs.io/). Via fsspec, we can thus interface dCache to a number of libraries: [Xarray](https://docs.xarray.dev) and [Zarr](https://zarr.dev/), but also [Rasterio](https://rasterio.readthedocs.io), [Pandas](https://pandas.pydata.org/), ... 

dCacheFS, like other fsspec implementations, can be configured via environment variables:

In [ ]:
import os
print(os.environ["FSSPEC_DCACHE_API_URL"])
print(os.environ["FSSPEC_DCACHE_WEBDAV_URL"])
print(os.environ["FSSPEC_DCACHE_TOKEN"])

Once imported, dCacheFS registers itself as a fsspec implementation for the `dcache://` protocol:

In [ ]:
import dcachefs
import fsspec

# create an filesystem instance 
fs = fsspec.filesystem("dcache")
fs

## 3. Example I - NetCDF/HDF5 files

We use fsspec API to create file-like objects:

In [ ]:
files = [
    fs.open(
        f"/daymet-daily-v4/region-na/na-{year}/daymet_v4_daily_na_tmax_{year}.nc", 
        block_size=1 * 2**20  # 1MB
    ) 
    for year in range(1980, 2020)
]
files

We load data from the objects, concatenating the datasets in a single "virtual" dataset. Metadata from each file is parsed in parallel using the Dask cluster:

In [ ]:
# Connect to Dask cluster to open data files in parallel

In [ ]:
import xarray as xr

In [ ]:
%%time
ds = xr.open_mfdataset(
    files, 
    engine="h5netcdf",
    compat="override",
    coords="all",
    parallel=True,
    data_vars=["tmax"],
    drop_variables=("lat", "lon"),
    chunks="auto",
)
ds

In [ ]:
ds["tmax"]

Select only one time slice and visualize its content:

In [ ]:
ds["tmax"].isel(time=0).plot.imshow()

## 4. Example II - Zarr store

Zarr stores can be seen as key-value maps - from chunk labels to actual data. fsspec allows to generate such mappings:

In [ ]:
mapper = fsspec.get_mapper("dcache://spring-index-models.zarr")

Open data from a subdirectory (group) in the store:

In [ ]:
spring_index_1980 = xr.open_zarr(mapper, group="1980")
spring_index_1980

Select only one variable and plant, and visualize its content:

In [ ]:
spring_index_1980["first-bloom"].sel(plant="lilac").plot.imshow()